In [4]:
import pandas as pd

# 1. Carregar o conjunto de dados do compressor de ar a partir de um ficheiro CSV
df = pd.read_csv("MetroPT3(AirCompressor).csv")

# 2. Remover a coluna "Unnamed: 0" que geralmente surge do índice guardado no CSV
df = df.drop(columns=["Unnamed: 0"])

# 3. Converter a coluna "timestamp" de texto (string) para o formato de data e hora (datetime) do Pandas
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 4. Ordenar os dados por ordem cronológica (do mais antigo para o mais recente) e reiniciar os índices
df = df.sort_values("timestamp").reset_index(drop=True)

# 5. Imprimir estatísticas básicas e inspeção inicial dos dados
print("Dimensão:", df.shape)                                    # Mostra o número de linhas e colunas
print("Período:", df["timestamp"].min(), "->", df["timestamp"].max())  # Mostra a data inicial e final do dataset
print("Tipo do timestamp:", df["timestamp"].dtype)              # Confirma se o tipo de dados do timestamp está correto
print(df.dtypes)                                                # Mostra o tipo de dados de todas as colunas
print("Valores em falta:", df.isna().sum().sum())               # Conta o total de valores nulos (NaN) em todo o DataFrame
print(df.describe().T)                                          # Mostra estatísticas descritivas (média, min, max, etc.) transpostas

Dimensão: (1516948, 16)
Período: 2020-02-01 00:00:00 -> 2020-09-01 03:59:50
Tipo do timestamp: datetime64[us]
timestamp          datetime64[us]
TP2                       float64
TP3                       float64
H1                        float64
DV_pressure               float64
Reservoirs                float64
Oil_temperature           float64
Motor_current             float64
COMP                      float64
DV_eletric                float64
Towers                    float64
MPG                       float64
LPS                       float64
Pressure_switch           float64
Oil_level                 float64
Caudal_impulses           float64
dtype: object
Valores em falta: 0
                     count                        mean                  min  \
timestamp          1516948  2020-05-16 22:58:36.631517  2020-02-01 00:00:00   
TP2              1516948.0                    1.367826               -0.032   
TP3              1516948.0                    8.984611                 0.73

In [10]:
from sklearn.preprocessing import MinMaxScaler

# 6. Períodos de avaria (confirmar na página da UCI, dataset 791, e citar no relatório)
falhas = [
    ("2020-04-18 00:00", "2020-04-18 23:59"),
    ("2020-05-29 23:30", "2020-05-30 06:00"),
    ("2020-06-05 10:00", "2020-06-07 14:30"),
    ("2020-07-15 14:30", "2020-07-15 19:00"),
]

# 7. Coluna alvo: 0 = funcionamento normal, 1 = falha
df["falha"] = 0
for inicio, fim in falhas:
    df.loc[df["timestamp"].between(inicio, fim), "falha"] = 1

# 8. Distribuição das classes
print("Distribuição das classes:")
print(df["falha"].value_counts())
print(df["falha"].value_counts(normalize=True) * 100)

Distribuição das classes:
falha
0    1486994
1      29954
Name: count, dtype: int64
falha
0    98.025377
1     1.974623
Name: proportion, dtype: float64


In [11]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# 9. Média por minuto; a falha fica 1 se algum instante do minuto for falha
agg = {c: "mean" for c in df.columns if c not in ["timestamp", "falha"]}
agg["falha"] = "max"
dados = df.set_index("timestamp").resample("1min").agg(agg).dropna().reset_index()
dados["falha"] = dados["falha"].astype(int)

colunas = [c for c in dados.columns if c not in ["timestamp", "falha"]]

# 10. DECISÃO TUA: onde cortar entre treino (passado) e teste (futuro)
PROP_TREINO = 0.5       # corte por proporção
CORTE_DATA = None       # ou, por exemplo, "2020-06-01" (entre a 2.ª e a 3.ª avaria)

if CORTE_DATA is None:
    corte = int(len(dados) * PROP_TREINO)
else:
    corte = int((dados["timestamp"] < pd.Timestamp(CORTE_DATA)).sum())

# 11. Divisão cronológica, sem sorteio
X_train, X_test = dados[colunas].iloc[:corte], dados[colunas].iloc[corte:]
y_train, y_test = dados["falha"].iloc[:corte], dados["falha"].iloc[corte:]

print("Treino:", len(X_train), "| falhas:", y_train.sum())
print("Teste: ", len(X_test), "| falhas:", y_test.sum())

Treino: 126360 | falhas: 1435
Teste:  126360 | falhas: 3523


In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, digits=3))

[[122515    322]
 [  1878   1645]]
              precision    recall  f1-score   support

           0      0.985     0.997     0.991    122837
           1      0.836     0.467     0.599      3523

    accuracy                          0.983    126360
   macro avg      0.911     0.732     0.795    126360
weighted avg      0.981     0.983     0.980    126360



In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras

np.random.seed(42)
keras.utils.set_random_seed(42)      # resultados reproduzíveis

JANELA, PASSO = 30, 5   # cada amostra = 30 minutos seguidos de sensores

# Normalização: o scaler aprende só com o treino (sem fuga de informação)
scaler = StandardScaler().fit(X_train)
Xtr = scaler.transform(X_train).astype("float32")
Xte = scaler.transform(X_test).astype("float32")

def janelas(X, y, janela, passo):
    y = np.asarray(y)
    idx = range(janela, len(X), passo)
    Xs = np.stack([X[i - janela:i] for i in idx])
    ys = np.array([y[i - 1] for i in idx])   # rótulo = último minuto da janela
    return Xs, ys

Xw_tr, yw_tr = janelas(Xtr, y_train, JANELA, PASSO)
Xw_te, yw_te = janelas(Xte, y_test, JANELA, PASSO)

# Pesos de classe (falhas são raras)
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=yw_tr)
class_weight = {0: pesos[0], 1: pesos[1]}

modelo = keras.Sequential([
    keras.layers.Input(shape=(JANELA, Xw_tr.shape[2])),
    keras.layers.LSTM(32),
    keras.layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy")
modelo.fit(Xw_tr, yw_tr, epochs=5, batch_size=256,
           class_weight=class_weight, verbose=1)

y_pred_lstm = (modelo.predict(Xw_te) > 0.5).astype(int).ravel()

print("LSTM")
print(confusion_matrix(yw_te, y_pred_lstm))
print(classification_report(yw_te, y_pred_lstm, digits=3))

# Comparação justa: a Random Forest avaliada nos mesmos instantes que o LSTM
idx = np.arange(JANELA, len(X_test), PASSO) - 1
print("RANDOM FOREST (mesmos instantes do LSTM)")
print(confusion_matrix(y_test.iloc[idx], y_pred_rf[idx]))
print(classification_report(y_test.iloc[idx], y_pred_rf[idx], digits=3))

Epoch 1/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.1785
Epoch 2/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0437
Epoch 3/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0375
Epoch 4/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0276
Epoch 5/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0332
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
LSTM
[[22829  1732]
 [   64   641]]
              precision    recall  f1-score   support

           0      0.997     0.929     0.962     24561
           1      0.270     0.909     0.417       705

    accuracy                          0.929     25266
   macro avg      0.634     0.919     0.689     25266
weighted avg      0.977     0.929     0.947     25266

RANDOM FOREST (mesmos instantes do LSTM)
[[24497    64]
 [  387   318]]
              precision    recall  f1-score   support

           0      0.984     0.997     0.991     24561
           1      0.832     0.451     0.585       705

    accuracy                          0.